# Scénario d'usage — Détection de *faulty* embarquée en environnement industriel

**Projet** : CL-Embedded · Apprentissage incrémental sur microcontrôleur (NUCLEO-F439ZI)
**But de ce notebook** : cadrer, sous forme de schémas, **comment la carte fonctionne sur site** — du
déploiement à la surveillance continue avec adaptation autonome (continual learning).

> Ce notebook est **volontairement autonome** (aucune dépendance aux expériences du dépôt) : il ne produit
> que des *schémas de contexte*. Il sert de **base réutilisable** pour d'autres visuels du même style —
> présentation, mise en valeur du produit pour la start-up.

Le scénario est décomposé en **3 schémas complémentaires** pour ne pas surcharger une seule figure :

1. **Diagramme de cas d'usage** (vue « produit » : qui fait quoi avec la carte)
2. **Cycle de vie / activité** (déploiement → récolte → entraînement → inférence, avec les boucles de décision)
3. **Zoom sur la boucle temps réel** (inférence + détection *faulty* **en parallèle de** la détection de drift → mise à jour CL)

Un dernier schéma **conceptuel** (normal / *faulty* / drift) clarifie la nuance la plus importante du scénario.


## S0 — Setup & helpers de dessin

On se replace à la racine du dépôt, on prépare le dossier de figures, et on définit une petite
**boîte à outils de schéma** (`box`, `diamond`, `oval`, `arrow`, `actor`) réutilisable pour tous les
plots suivants — c'est elle qu'on recyclera pour les futurs visuels de contexte.

In [1]:
from pathlib import Path
import os, sys

import matplotlib
matplotlib.use("Agg")  # backend non interactif (exécution nbconvert)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Ellipse, Polygon, Circle, Rectangle
import numpy as np
from IPython.display import Markdown, display

# --- racine dépôt (robuste : depuis notebooks/ ou racine) ---
_cwd = Path(".").resolve()
if _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
FIGURE_DIR = REPO_ROOT / "docs" / "figures" / "scenario_usecase"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "savefig.bbox": "tight"})

# --- palette (professionnelle, cohérente entre schémas) ---
C = {
    "deploy":   "#cfe8ff",  # bleu   — déploiement / matériel
    "collect":  "#ffe8b3",  # ambre  — récolte de données
    "train":    "#e6d7f4",  # violet — entraînement / calibration
    "infer":    "#d7f4d7",  # vert   — inférence normale
    "alert":    "#ffd0d0",  # rouge  — alerte faulty
    "adapt":    "#ffdcc0",  # orange — adaptation / MAJ CL
    "decision": "#fff3c4",  # jaune  — décision (losange)
    "actor":    "#f3f3f3",  # gris   — acteur externe
    "edge":     "#333333",
}

def save(fig, name):
    out = FIGURE_DIR / name
    fig.savefig(out, dpi=150)
    print("✓", out.relative_to(REPO_ROOT))

def box(ax, x, y, w, h, label, fc="#eee", ec=C["edge"], fontsize=10, lw=1.6, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle="round,pad=0.02,rounding_size=0.08", fc=fc, ec=ec, lw=lw))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center",
            fontsize=fontsize, fontweight="bold" if bold else "normal")

def oval(ax, cx, cy, w, h, label, fc, ec=C["edge"], fontsize=9):
    ax.add_patch(Ellipse((cx, cy), w, h, fc=fc, ec=ec, lw=1.4))
    ax.text(cx, cy, label, ha="center", va="center", fontsize=fontsize)

def diamond(ax, cx, cy, w, h, label, fc=None, ec="#c9a227", fontsize=9):
    fc = fc or C["decision"]
    pts = [(cx, cy + h/2), (cx + w/2, cy), (cx, cy - h/2), (cx - w/2, cy)]
    ax.add_patch(Polygon(pts, closed=True, fc=fc, ec=ec, lw=1.6))
    ax.text(cx, cy, label, ha="center", va="center", fontsize=fontsize)

def arrow(ax, p0, p1, color=None, lw=1.8, label=None, style="-|>",
          ls="-", rad=0.0, txt=(0, 0.14), fontsize=8.5, fstyle="normal"):
    color = color or C["edge"]
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style, mutation_scale=15,
                 lw=lw, color=color, linestyle=ls,
                 connectionstyle=f"arc3,rad={rad}"))
    if label:
        mx = (p0[0] + p1[0]) / 2 + txt[0]
        my = (p0[1] + p1[1]) / 2 + txt[1]
        ax.text(mx, my, label, ha="center", va="center",
                fontsize=fontsize, color=color, style=fstyle)

def actor(ax, x, y, name, fontsize=9.5):
    ax.add_patch(Circle((x, y + 0.55), 0.17, fc="#fff", ec=C["edge"], lw=1.6))
    ax.plot([x, x], [y + 0.38, y - 0.30], color=C["edge"], lw=1.6)
    ax.plot([x - 0.27, x + 0.27], [y + 0.12, y + 0.12], color=C["edge"], lw=1.6)
    ax.plot([x, x - 0.24], [y - 0.30, y - 0.62], color=C["edge"], lw=1.6)
    ax.plot([x, x + 0.24], [y - 0.30, y - 0.62], color=C["edge"], lw=1.6)
    ax.text(x, y - 0.92, name, ha="center", va="center", fontsize=fontsize, fontweight="bold")

print("REPO_ROOT :", REPO_ROOT)
print("Figures   :", FIGURE_DIR.relative_to(REPO_ROOT))


REPO_ROOT : /home/leonard/Documents/ENAC/cl-embedded
Figures   : docs/figures/scenario_usecase


## S1 — Diagramme de cas d'usage (vue « produit »)

Qui interagit avec la carte, et pour quoi faire. Trois acteurs externes :

- **Équipement / Machine** — la source des mesures capteurs (la chose surveillée) ;
- **Technicien / Exploitant** — déploie la carte, déclenche la calibration, et *peut* fournir un label (apprentissage actif) ;
- **Serveur / Supervision** — reçoit les alertes.

La frontière du système = **la carte embarquée**. Les flèches en pointillé `«include»` montrent qu'une
détection *faulty* **déclenche** l'alerte, et qu'une détection de **drift** **déclenche** la mise à jour CL.

In [2]:
fig, ax = plt.subplots(figsize=(12.5, 7.2)); ax.axis("off")

# --- frontière système : la carte ---
box(ax, 4.2, 0.5, 5.2, 6.5, "", fc="#ffffff", ec="#5a5a5a", lw=2.0)
ax.text(6.8, 6.72, "Carte embarquée — NUCLEO-F439ZI",
        ha="center", va="center", fontsize=11.5, fontweight="bold", color="#333")

# --- cas d'usage (ovales) : 2 colonnes x 3 lignes ---
xL, xR = 5.55, 8.05
rows = [5.75, 4.15, 2.55]
oval(ax, xL, rows[0], 2.1, 1.0, "Collecter données\n(fonctionnement normal)", C["collect"])
oval(ax, xR, rows[0], 2.1, 1.0, "Entraîner / calibrer\nle modèle", C["train"])
oval(ax, xL, rows[1], 2.1, 1.0, "Détecter « faulty »\n(inférence continue)", C["infer"])
oval(ax, xR, rows[1], 2.1, 1.0, "Détecter le drift\n(à chaque donnée)", C["adapt"])
oval(ax, xL, rows[2], 2.1, 1.0, "Alerter\n(LED + signal serveur)", C["alert"])
oval(ax, xR, rows[2], 2.1, 1.0, "Mettre à jour le\nmodèle (CL)", C["adapt"])

# --- acteurs ---
actor(ax, 1.6, 5.1, "Équipement /\nMachine")
actor(ax, 1.6, 2.3, "Technicien /\nExploitant")
actor(ax, 11.4, 4.1, "Serveur /\nSupervision")

L = "#5a5a5a"
# Équipement -> fournit les mesures capteurs
for cy in rows:
    arrow(ax, (2.0, 5.1), (xL - 1.05, cy), color=L, style="-", lw=1.3, rad=0.05)
ax.text(2.9, 5.55, "mesures capteurs", fontsize=8.5, color=L, style="italic")
# Technicien -> calibration & MAJ (label optionnel)
arrow(ax, (2.0, 2.6), (xR - 1.05, rows[0]), color=L, style="-", lw=1.3, rad=-0.15)
arrow(ax, (2.0, 2.2), (xR - 1.05, rows[2]), color=L, style="-", lw=1.3, rad=0.05)
ax.text(3.0, 2.0, "déclenche / label (optionnel)", fontsize=8.5, color=L, style="italic")
# Alerter -> serveur
arrow(ax, (xL + 1.05, rows[2]), (11.0, 3.7), color="#c0392b", style="-|>", lw=1.6, rad=-0.12)
ax.text(9.9, 2.5, "alerte", fontsize=8.5, color="#c0392b", style="italic")

# --- include (déclenchements internes, pointillés) ---
arrow(ax, (xL, rows[1] - 0.5), (xL, rows[2] + 0.5), color="#b06a00", ls=(0,(4,3)),
      label="«include»", txt=(0.85, 0), fontsize=8, fstyle="italic")
arrow(ax, (xR, rows[1] - 0.5), (xR, rows[2] + 0.5), color="#b06a00", ls=(0,(4,3)),
      label="«include»", txt=(0.85, 0), fontsize=8, fstyle="italic")

ax.set_xlim(0.2, 12.6); ax.set_ylim(1.0, 7.2)
ax.set_title("Cas d'usage — surveillance de « faulty » avec adaptation embarquée",
             fontsize=13, pad=12)
save(fig, "01_use_case.png"); plt.close(fig)
display(Markdown("![use case](../docs/figures/scenario_usecase/01_use_case.png)"))


✓ docs/figures/scenario_usecase/01_use_case.png


![use case](../docs/figures/scenario_usecase/01_use_case.png)

## S2 — Cycle de vie : déploiement → récolte → entraînement → inférence

Le déroulé chronologique avec ses **deux boucles de décision** :

- *« Assez de données ? »* — sinon on continue la récolte ;
- *« Le modèle converge ? »* — sinon on **retourne récolter** davantage de données normales.

Une fois le modèle **satisfaisant**, on bascule en **mode inférence** (détaillé au schéma S3).

In [3]:
fig, ax = plt.subplots(figsize=(12.5, 6.6)); ax.axis("off")

# colonne centrale, on descend
cx = 4.4
box(ax, cx-1.3, 8.4, 2.6, 0.8, "Carte + capteurs alimentés", fc=C["deploy"], bold=True)
box(ax, cx-1.6, 6.9, 3.2, 0.95,
    "① Déploiement\nCarte placée sur site, capteurs OK", fc=C["deploy"])
box(ax, cx-1.6, 5.3, 3.2, 0.95,
    "② Récolte de données\nmachine en fonctionnement normal", fc=C["collect"])
diamond(ax, cx, 3.9, 2.5, 1.0, "Assez de données ?\n(image du normal)")
box(ax, cx-1.6, 2.3, 3.2, 0.95,
    "③ Entraînement / calibration\ndu modèle", fc=C["train"])
diamond(ax, cx, 0.9, 2.3, 1.0, "Le modèle\nconverge ?")
box(ax, 9.0, 0.4, 2.9, 1.0, "④ Mode inférence\n(→ voir schéma S3)", fc=C["infer"], bold=True)

# flux principal
arrow(ax, (cx, 8.4), (cx, 7.87))
arrow(ax, (cx, 6.9), (cx, 6.28))
arrow(ax, (cx, 5.3), (cx, 4.42))
arrow(ax, (cx, 3.4), (cx, 3.28), label="oui", txt=(0.35, 0.05))
arrow(ax, (cx, 2.3), (cx, 1.42))
arrow(ax, (cx+1.15, 0.9), (9.0, 0.9), label="oui — satisfaisant", txt=(0, 0.26), color="#2e7d32")

# boucle "pas assez de données" : diamant -> retour récolte (gauche, ambre)
arrow(ax, (cx-1.25, 3.9), (1.8, 3.9), color="#b06a00", label="non", txt=(0, 0.22))
arrow(ax, (1.8, 3.9), (1.8, 5.77), color="#b06a00", style="-", lw=1.8)
arrow(ax, (1.8, 5.77), (cx-1.6, 5.77), color="#b06a00")
ax.text(1.35, 4.9, "récolter\nplus", fontsize=8.5, color="#b06a00", ha="center")

# boucle "ne converge pas" : grand retour à ② (droite, rouge, contourne ④)
ax.text(cx+0.75, 0.6, "non", fontsize=8.5, color="#c0392b")
arrow(ax, (cx, 0.4), (cx, 0.1), color="#c0392b", style="-", lw=1.8)
arrow(ax, (cx, 0.1), (12.2, 0.1), color="#c0392b", style="-", lw=1.8)
arrow(ax, (12.2, 0.1), (12.2, 5.77), color="#c0392b", style="-", lw=1.8)
arrow(ax, (12.2, 5.77), (cx+1.6, 5.77), color="#c0392b")
ax.text(12.05, 3.2, "retour ② —\nrécolter plus de\ndonnées normales",
        fontsize=8.5, color="#c0392b", ha="right")

ax.set_xlim(0.0, 13.0); ax.set_ylim(-0.3, 9.6)
ax.set_title("Cycle de vie de la carte sur site (étapes ① → ④)", fontsize=13, pad=10)
save(fig, "02_cycle_de_vie.png"); plt.close(fig)
display(Markdown("![cycle](../docs/figures/scenario_usecase/02_cycle_de_vie.png)"))


✓ docs/figures/scenario_usecase/02_cycle_de_vie.png


![cycle](../docs/figures/scenario_usecase/02_cycle_de_vie.png)

## S3 — Boucle temps réel : détection *faulty* **∥** détection de drift → MAJ CL

En mode inférence, **à chaque donnée reçue**, deux traitements tournent **en parallèle** :

- **Voie détection** — l'inférence classe la donnée ; si *faulty* → **flash LED** (+ signal serveur si possible) ;
- **Voie adaptation** — la détection de drift surveille la distribution ; si drift → **mise à jour CL**
  (selon le modèle : récolter davantage, ou MAJ continue sur une fenêtre), puis retour à l'inférence normale.

La barre `∥` (*fork/join*) marque le parallélisme : les deux voies repartent de la même acquisition.

In [4]:
fig, ax = plt.subplots(figsize=(12.5, 7.4)); ax.axis("off")

GRN = "#2e7d32"

# acquisition + fork
box(ax, 5.0, 6.6, 3.0, 0.8, "Nouvelle donnée capteur", fc=C["deploy"], bold=True)
ax.add_patch(Rectangle((4.4, 6.15), 4.2, 0.14, fc="#333"))  # barre fork (parallélisme)
arrow(ax, (6.5, 6.6), (6.5, 6.31))

# ---- voie gauche : détection faulty ----
xg = 3.0
arrow(ax, (5.1, 6.22), (xg + 0.4, 5.55), rad=0.05)
box(ax, xg-1.5, 4.75, 3.0, 0.8, "Inférence\n(classer la donnée)", fc=C["infer"])
arrow(ax, (xg, 4.75), (xg, 3.95))
diamond(ax, xg, 3.4, 2.4, 1.05, "« faulty » ?")
box(ax, xg-1.6, 1.6, 3.2, 0.85, "Flash LED\n+ signal serveur", fc=C["alert"], bold=True)
arrow(ax, (xg, 2.87), (xg, 2.47), label="oui", txt=(0.32, 0.02), color="#c0392b")

# ---- voie droite : détection de drift ----
xd = 9.6
arrow(ax, (7.9, 6.22), (xd - 0.4, 5.55), rad=-0.05)
box(ax, xd-1.6, 4.75, 3.2, 0.8, "Détection de drift\n(à chaque donnée)", fc=C["adapt"])
arrow(ax, (xd, 4.75), (xd, 3.95))
diamond(ax, xd, 3.4, 2.4, 1.05, "drift\ndétecté ?")
arrow(ax, (xd, 2.87), (xd, 2.62), label="oui", txt=(0.35, 0.02), color="#c0392b")
box(ax, xd-1.7, 1.55, 3.4, 1.05,
    "Mise à jour CL\n(label vrai / pseudo-label ;\nfenêtre ou récolte ciblée)", fc=C["adapt"], fontsize=8.5)
arrow(ax, (xd, 1.55), (xd, 1.05))
diamond(ax, xd, 0.55, 2.4, 0.95, "modèle\nadapté ?")

# ---- retours (join) : rails verts remontant vers l'acquisition ----
# rail gauche : faulty=non  +  après alerte
arrow(ax, (xg-1.2, 3.4), (0.6, 3.4), color=GRN, style="-", lw=1.5, label="non", txt=(0, 0.22))
arrow(ax, (xg, 1.6), (xg, 1.2), color=GRN, style="-", lw=1.5)
arrow(ax, (xg, 1.2), (0.6, 1.2), color=GRN, style="-", lw=1.5)
arrow(ax, (0.6, 1.2), (0.6, 7.0), color=GRN, style="-", lw=1.5)
arrow(ax, (0.6, 7.0), (4.98, 7.0), color=GRN, style="-|>", lw=1.5)

# rail droit : drift=non  +  modèle adapté=oui
arrow(ax, (xd+1.2, 3.4), (12.4, 3.4), color=GRN, style="-", lw=1.5, label="non", txt=(0, 0.22))
arrow(ax, (xd+1.2, 0.55), (12.4, 0.55), color=GRN, style="-", lw=1.5, label="oui", txt=(0, 0.22))
arrow(ax, (12.4, 0.55), (12.4, 7.0), color=GRN, style="-", lw=1.5)
arrow(ax, (12.4, 7.0), (8.02, 7.0), color=GRN, style="-|>", lw=1.5)

# modèle adapté = non -> re-boucle de mise à jour
arrow(ax, (xd-1.2, 0.55), (7.0, 0.55), color="#c0392b", style="-", lw=1.4, label="non", txt=(0, 0.22))
arrow(ax, (7.0, 0.55), (7.0, 2.075), color="#c0392b", style="-", lw=1.4)
arrow(ax, (7.0, 2.075), (xd-1.7, 2.075), color="#c0392b", style="-|>", lw=1.4)

ax.text(6.5, -0.4, "les deux voies repartent de « Nouvelle donnée capteur » — boucle continue",
        ha="center", fontsize=9, style="italic", color="#555")

ax.set_xlim(0.0, 13.0); ax.set_ylim(-0.7, 7.7)
ax.set_title("Mode inférence — détection « faulty » en parallèle de l'adaptation au drift",
             fontsize=13, pad=10)
save(fig, "03_boucle_temps_reel.png"); plt.close(fig)
display(Markdown("![boucle](../docs/figures/scenario_usecase/03_boucle_temps_reel.png)"))


✓ docs/figures/scenario_usecase/03_boucle_temps_reel.png


![boucle](../docs/figures/scenario_usecase/03_boucle_temps_reel.png)

## S4 — Schéma conceptuel : *normal* vs *faulty* vs *drift*

La nuance la plus importante du scénario (et la vraie difficulté). Le modèle apprend une **image du
fonctionnement normal**. Toute donnée qui s'en écarte est une *déviation* — mais il faut décider si c'est :

- une **panne naissante** (*faulty*) → **alerter** (ne surtout pas « apprendre » comme normal) ;
- un **changement légitime de régime** (drift : nouvel équipement, saison, usure lente) → **s'adapter** (MAJ CL).

Cette figure illustre les trois cas dans un espace de features simplifié.

In [5]:
rng = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(9.5, 6.2))

normal = rng.normal([0, 0], 0.6, size=(200, 2))
drift  = rng.normal([2.6, 1.4], 0.6, size=(120, 2))
faulty = rng.normal([-0.3, 3.1], 0.45, size=(25, 2))

ax.scatter(normal[:,0], normal[:,1], s=22, c="#2e7d32", alpha=0.55, label="Normal (appris)")
ax.scatter(drift[:,0],  drift[:,1],  s=22, c="#e07b00", alpha=0.6,
           marker="s", label="Drift — nouveau régime → s'adapter (CL)")
ax.scatter(faulty[:,0], faulty[:,1], s=55, c="#c0392b", marker="X",
           label="Faulty — anomalie rare → alerter")

# frontière "normal" apprise
from matplotlib.patches import Ellipse as _E
ax.add_patch(_E((0,0), 3.0, 3.0, fill=False, ec="#2e7d32", lw=1.6, ls="--"))
ax.text(0, -1.85, "frontière du « normal »\napprise à la calibration",
        ha="center", fontsize=8.5, color="#2e7d32")
ax.annotate("s'adapter\n(le régime a changé,\ndonnées denses & durables)",
            (2.6, 1.4), (3.6, -0.4), fontsize=8.5, color="#b06a00",
            arrowprops=dict(arrowstyle="->", color="#b06a00"))
ax.annotate("alerter\n(rare, isolé,\npotentielle panne)",
            (-0.3, 3.1), (-3.7, 2.4), fontsize=8.5, color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b"))

ax.set_xlabel("feature 1 (ex. vibration RMS)"); ax.set_ylabel("feature 2 (ex. température)")
ax.set_title("Décision clé : une déviation est-elle un « faulty » (alerter)\n"
             "ou un « drift » (apprendre) ?", fontsize=12)
ax.legend(loc="upper right", fontsize=8.5, framealpha=0.95)
ax.set_xlim(-4.5, 5.5); ax.set_ylim(-2.5, 4.5); ax.grid(alpha=0.15)
save(fig, "04_concept_normal_faulty_drift.png"); plt.close(fig)
display(Markdown("![concept](../docs/figures/scenario_usecase/04_concept_normal_faulty_drift.png)"))


✓ docs/figures/scenario_usecase/04_concept_normal_faulty_drift.png


![concept](../docs/figures/scenario_usecase/04_concept_normal_faulty_drift.png)

## S5 — Est-ce que le scénario « fait du sens » ?

**Oui, globalement le scénario est cohérent et bien posé.** Il correspond exactement à la logique déjà
implémentée dans le projet (détection non-supervisée type Mahalanobis calibrée sur le *normal* +
mise à jour EWC déclenchée par un *gate* de nouveauté/drift — cf. Sprint 38). Points forts :

- ✅ **Calibration sur le normal uniquement** (étape ②) = hypothèse réaliste et standard en maintenance
  prédictive : on ne dispose presque jamais d'exemples de pannes au déploiement.
- ✅ **Boucle « converge ? → récolter plus »** = correcte : un détecteur mal calibré vient d'un échantillon
  de normal trop court ou non représentatif.
- ✅ **Détection + adaptation en parallèle** = c'est bien le cœur du continual learning embarqué.

**3 nuances à trancher avec Frédéric** (ce sont des *décisions produit*, pas des bugs) :

1. **Faulty ≠ drift — c'est LA difficulté** (schéma S4). Les deux sont des écarts au normal. Or si on
   « apprend » une panne naissante comme un nouveau normal, on **s'aveugle** (on annule l'alerte future).
   → Règle de sûreté suggérée : un écart **rare/isolé/transitoire** = *faulty* (alerter) ; un écart
   **dense/persistant/durable** = drift (adapter). La MAJ CL ne doit s'appliquer **que** sur du drift confirmé.

2. **Label de la MAJ CL** : en autonomie totale on met à jour en **pseudo-label** (le modèle se croit) —
   risque de dérive silencieuse. En **apprentissage actif**, le technicien valide le label (plus sûr,
   moins autonome). Le projet supporte les deux (politiques *gated_pseudolabel* / *gated_truelabel*).

3. **« Modèle adapté ? »** doit avoir un critère mesurable (score de drift repassé sous le seuil, ou
   fenêtre de N points stables) pour éviter une MAJ qui ne s'arrête jamais.

**Petite reformulation du flux** (pour être exact) : la « détection de drift » et la « MAJ CL » gagnent à
être séparées de la détection *faulty* — c'est ce que montre S3 : deux voies parallèles qui **repartent
toutes deux** de l'acquisition suivante.


## S6 — Autres schémas du même style (propositions réutilisables)

Idées de visuels de **contexte / mise en valeur produit**, réalisables avec les mêmes helpers `S0` :

| # | Schéma | Message porté (pour la start-up) |
|---|--------|----------------------------------|
| A | **Edge vs Cloud** (avant/après) : capteur→cloud→modèle vs capteur→carte (tout local) | « pas de connexion permanente, faible latence, données qui ne quittent pas le site » |
| B | **Budget latence** (barres) : inférence · inférence+MAJ · **budget 100 ms** (Gap 2) | « on tient largement le temps réel » — chiffres réels mesurés board |
| C | **Budget RAM** (donut) : `.bss` utilisé vs 256 Ko SRAM | « tient dans un microcontrôleur standard, sans NPU » |
| D | **Économie des mises à jour** : MAJ déclenchées (gated) vs permanentes (always) | « adaptation autonome **sans** gaspiller l'énergie » (Sprint 38) |
| E | **Timeline de déploiement** (Gantt) : installation → calibration → opérationnel | « mise en service rapide, peu d'intervention humaine » |
| F | **Chaîne de valeur** : signal capteur → features → détection → décision → alerte | vue « pipeline produit » de bout en bout |

Ci-dessous, **deux exemples implémentés** (B et C) pour amorcer la réutilisation.
Les valeurs sont ici des **placeholders illustratifs** — à remplacer par les mesures réelles du dépôt
(`docs/triple_gap.md`, `exp_S3x_*`) avant tout usage externe.

In [6]:
# --- Exemple B : budget latence (placeholders illustratifs) ---
fig, ax = plt.subplots(figsize=(9, 3.6))
labels = ["Inférence seule", "Inférence + MAJ CL", "Budget temps réel"]
vals   = [0.08, 0.34, 100.0]            # ms — À REMPLACER par mesures board réelles
cols   = [C["infer"], C["adapt"], "#dddddd"]
bars = ax.barh(labels, vals, color=cols, ec="#333")
ax.set_xscale("log"); ax.set_xlabel("Latence par échantillon (ms, échelle log)")
for b, v in zip(bars, vals):
    ax.text(v*1.1, b.get_y()+b.get_height()/2, f"{v:g} ms", va="center", fontsize=9)
ax.axvline(100, color="#c0392b", ls="--", lw=1.2)
ax.text(100, 2.4, "Gap 2 : ≤ 100 ms", color="#c0392b", fontsize=9, ha="center")
ax.set_title("Budget de latence — placeholders (remplacer par mesures réelles)", fontsize=11)
ax.set_xlim(0.01, 300)
save(fig, "B_budget_latence.png"); plt.close(fig)
display(Markdown("![B](../docs/figures/scenario_usecase/B_budget_latence.png)"))


✓ docs/figures/scenario_usecase/B_budget_latence.png


![B](../docs/figures/scenario_usecase/B_budget_latence.png)

In [7]:
# --- Exemple C : budget RAM (placeholder illustratif) ---
fig, ax = plt.subplots(figsize=(5.6, 5.6))
used = 105.0            # Ko .bss — À REMPLACER (ex. Sprint 34/38 ~105 Ko)
total = 256.0
ax.pie([used, total-used], labels=[f"Utilisé\n{used:g} Ko", f"Libre\n{total-used:g} Ko"],
       colors=[C["infer"], "#eeeeee"], startangle=90, counterclock=False,
       wedgeprops=dict(ec="#333", lw=1.2), autopct=lambda p: f"{p:.0f}%")
ax.set_title(f"Empreinte RAM sur {total:g} Ko SRAM (NUCLEO-F439ZI)\n"
             "placeholder — remplacer par .bss réel", fontsize=11)
save(fig, "C_budget_ram.png"); plt.close(fig)
display(Markdown("![C](../docs/figures/scenario_usecase/C_budget_ram.png)"))


✓ docs/figures/scenario_usecase/C_budget_ram.png


![C](../docs/figures/scenario_usecase/C_budget_ram.png)

---
*Notebook autonome — schémas de contexte / produit. Régénérer les figures via
`jupyter nbconvert --to notebook --execute notebooks/scenario_usecase_industriel.ipynb`.
Les figures sont écrites dans `docs/figures/scenario_usecase/`.*